In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
TRAIN_PATH = "../data/processed/train.csv"
VAL_PATH = "../data/processed/validation.csv"
TEST_PATH = "../data/processed/test.csv"

TARGET = "Class"

# Load the dataset

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (198608, 34)
Validation shape: (42559, 34)
Test shape: (42559, 34)


In [4]:
train_df.head()

,Time,Amount,Amount_log,V1,V2,V3,V4,V5,V6,V7,...,V22,V23,V24,V25,V26,V27,V28,Hour_cos,Hour_sin,Class
0,-0.564974,-0.306612,-0.270803,1.132089,0.194386,0.362150,0.588889,-0.271681,-0.398920,-0.099188,...,-0.603157,0.165303,0.174239,0.068777,0.094035,-0.014924,0.020603,-1.836970e-16,-1.000000e+00,0
1,-0.922877,0.460786,1.295971,-0.353027,-0.955101,1.715869,-1.862038,-1.993162,-0.171555,-0.058185,...,0.544119,0.334098,0.508283,-0.072215,-0.179250,0.007009,0.037677,-8.660254e-01,-5.000000e-01,0
2,0.211545,-0.355583,-1.218108,-0.646948,0.994754,2.041892,-0.318653,0.402454,0.073120,0.533261,...,-0.553597,-0.395771,-0.756263,0.332744,-0.713647,-0.353967,-0.151089,2.588190e-01,9.659258e-01,0
3,0.740763,0.022526,0.842189,-4.049534,-0.342911,-2.343460,-0.124219,2.275074,1.049156,1.202888,...,0.452424,0.251046,-1.079154,-0.377491,-0.347974,-0.903836,2.017586,-1.000000e+00,1.224647e-16,0
4,1.425136,-0.257518,0.081895,1.940494,-0.501533,0.047406,0.595139,-0.976805,-0.395403,-0.808078,...,0.186586,0.283139,-0.102873,-0.526160,0.173262,0.004812,-0.032236,7.071068e-01,-7.071068e-01,0


In [5]:
print("Target exists in train:", TARGET in train_df.columns)
print("Target exists in validation:", TARGET in val_df.columns)
print("Target exists in test:", TARGET in test_df.columns)

Target exists in train: True
Target exists in validation: True
Target exists in test: True


In [6]:
print("Train columns == Validation columns:",
      list(train_df.columns) == list(val_df.columns))

print("Train columns == Test columns:",
      list(train_df.columns) == list(test_df.columns))

Train columns == Validation columns: True
Train columns == Test columns: True


In [7]:
print("Train class distribution:")
print(train_df[TARGET].value_counts())

print("\nValidation class distribution:")
print(val_df[TARGET].value_counts())

print("\nTest class distribution:")
print(test_df[TARGET].value_counts())

Train class distribution:
Class
0    198277
1       331
Name: count, dtype: int64

Validation class distribution:
Class
0    42488
1       71
Name: count, dtype: int64

Test class distribution:
Class
0    42488
1       71
Name: count, dtype: int64


In [8]:
print("Train class percentage:")
print(train_df[TARGET].value_counts(normalize=True) * 100)

print("\nValidation class percentage:")
print(val_df[TARGET].value_counts(normalize=True) * 100)

print("\nTest class percentage:")
print(test_df[TARGET].value_counts(normalize=True) * 100)

Train class percentage:
Class
0    99.83334
1     0.16666
Name: proportion, dtype: float64

Validation class percentage:
Class
0    99.833173
1     0.166827
Name: proportion, dtype: float64

Test class percentage:
Class
0    99.833173
1     0.166827
Name: proportion, dtype: float64


In [9]:
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (198608, 33)
y_train: (198608,)
X_val: (42559, 33)
y_val: (42559,)
X_test: (42559, 33)
y_test: (42559,)


## import XGBoost

In [11]:
!pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   - -------------------------------------- 1.3/48.9 MB 3.2 MB/s eta 0:00:15
   - -------------------------------------- 2.1/48.9 MB 3.4 MB/s eta 0:00:14
   -- ------------------------------------- 2.9/48.9 MB 3.5 MB/s eta 0:00:14
   -- ------------------------------------- 3.7/48.9 MB 3.5 MB/s eta 0:00:13
   --- ------------------------------------ 4.2/48.9 MB 3.4 MB/s eta 0:00:13
   ---- ----------------------------------- 5.0/48.9 MB 3.4 MB/s eta 0:00:13
   ---- ----------------------------------- 5.2/48.9 MB 3.2 MB/s eta 0:00:14
   ---- ----------------------------------- 5.5/48.9 MB 3.0 MB/s eta 0:00:15
   ---- ----------------------------------- 5.8/48.9 MB 2.8 MB/s eta 0:00:16
   ---- ----------------------------------- 5.8/48.9 MB 2.8 MB/s eta 0:00:16
   ---- ------------

In [12]:
from xgboost import XGBClassifier

### Create the Baseline Model

In [13]:
baseline_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

In [14]:
#train the baseline model
baseline_model.fit(
    X_train,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [15]:
y_val_proba = baseline_model.predict_proba(X_val)[:, 1]

In [16]:
print(y_val_proba[:10])
print("Min probability:", y_val_proba.min())
print("Max probability:", y_val_proba.max())

[6.5394623e-08 1.8768943e-06 2.9377217e-05 9.9151862e-07 1.9288279e-06
 2.5409674e-06 3.4996945e-05 3.0929968e-06 1.9209627e-05 6.1524673e-07]
Min probability: 1.171176e-08
Max probability: 0.99999726


In [17]:
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [18]:
pr_auc = average_precision_score(y_val, y_val_proba)

print("Baseline PR-AUC:", pr_auc)

Baseline PR-AUC: 0.8679836250000378


In [19]:
threshold = 0.5

y_val_pred = (y_val_proba >= threshold).astype(int)

In [20]:
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Precision: 0.9830508474576272
Recall: 0.8169014084507042
F1-score: 0.8923076923076924


In [21]:
cm = confusion_matrix(y_val, y_val_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[42487     1]
 [   13    58]]


In [22]:
print("===== BASELINE RESULTS =====")
print(f"PR-AUC   : {pr_auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("\nConfusion Matrix:")
print(cm)

===== BASELINE RESULTS =====
PR-AUC   : 0.8680
Precision: 0.9831
Recall   : 0.8169
F1-score : 0.8923

Confusion Matrix:
[[42487     1]
 [   13    58]]


In [23]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative samples:", negative_count)
print("Positive samples:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

Negative samples: 198277
Positive samples: 331
scale_pos_weight: 599.02416918429


In [24]:
#weighted XGBoost
weighted_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

In [25]:
weighted_model.fit(
    X_train,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [26]:
y_val_proba_weighted = weighted_model.predict_proba(X_val)[:, 1]

In [27]:
threshold = 0.5

y_val_pred_weighted = (
    y_val_proba_weighted >= threshold
).astype(int)

In [28]:
#Calculate metrics for weighted model
pr_auc_weighted = average_precision_score(
    y_val,
    y_val_proba_weighted
)

precision_weighted = precision_score(
    y_val,
    y_val_pred_weighted
)

recall_weighted = recall_score(
    y_val,
    y_val_pred_weighted
)

f1_weighted = f1_score(
    y_val,
    y_val_pred_weighted
)

cm_weighted = confusion_matrix(
    y_val,
    y_val_pred_weighted
)

print("===== WEIGHTED XGBOOST =====")
print(f"PR-AUC   : {pr_auc_weighted:.4f}")
print(f"Precision: {precision_weighted:.4f}")
print(f"Recall   : {recall_weighted:.4f}")
print(f"F1-score : {f1_weighted:.4f}")
print("\nConfusion Matrix:")
print(cm_weighted)

===== WEIGHTED XGBOOST =====
PR-AUC   : 0.8737
Precision: 0.9365
Recall   : 0.8310
F1-score : 0.8806

Confusion Matrix:
[[42484     4]
 [   12    59]]


In [29]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline XGBoost",
        "Weighted XGBoost"
    ],
    "PR-AUC": [
        pr_auc,
        pr_auc_weighted
    ],
    "Precision": [
        precision,
        precision_weighted
    ],
    "Recall": [
        recall,
        recall_weighted
    ],
    "F1-score": [
        f1,
        f1_weighted
    ]
})

comparison

,Model,PR-AUC,Precision,Recall,F1-score
0,Baseline XGBoost,0.867984,0.983051,0.816901,0.892308
1,Weighted XGBoost,0.873673,0.936508,0.830986,0.880597


## Baseline vs Weighted XGBoost

The dataset is severely imbalanced, with approximately 599 legitimate
transactions for every fraudulent transaction in the training set.

The baseline XGBoost model achieved a validation PR-AUC of 0.8680,
while the class-weighted model using `scale_pos_weight ≈ 599`
achieved a PR-AUC of 0.8737.

At the default threshold of 0.5, class weighting increased recall
from 0.8169 to 0.8310, while precision decreased from 0.9831 to
0.9365. F1-score decreased from 0.8923 to 0.8806.

This indicates that class weighting made the model slightly more
sensitive to fraudulent transactions, at the cost of additional
false positives.

# Threshold Analysis

In [30]:
thresholds = np.arange(0.1, 1.0, 0.05)

threshold_results = []

for threshold in thresholds:
    y_pred = (y_val_proba_weighted >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1": f1_score(y_val, y_pred, zero_division=0),
        "False Positives": ((y_val == 0) & (y_pred == 1)).sum(),
        "False Negatives": ((y_val == 1) & (y_pred == 0)).sum(),
        "True Positives": ((y_val == 1) & (y_pred == 1)).sum()
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df

,Threshold,Precision,Recall,F1,False Positives,False Negatives,True Positives
0,0.10,0.857143,0.845070,0.851064,10,11,60
1,0.15,0.880597,0.830986,0.855072,8,12,59
2,0.20,0.893939,0.830986,0.861314,7,12,59
3,0.25,0.921875,0.830986,0.874074,5,12,59
4,0.30,0.921875,0.830986,0.874074,5,12,59
5,0.35,0.936508,0.830986,0.880597,4,12,59
6,0.40,0.936508,0.830986,0.880597,4,12,59
7,0.45,0.936508,0.830986,0.880597,4,12,59
8,0.50,0.936508,0.830986,0.880597,4,12,59
9,0.55,0.951613,0.830986,0.887218,3,12,59


In [31]:
threshold_results_df[
    ["Threshold", "Precision", "Recall", "F1",
     "False Positives", "False Negatives", "True Positives"]
].round(4)

,Threshold,Precision,Recall,F1,False Positives,False Negatives,True Positives
0,0.10,0.8571,0.8451,0.8511,10,11,60
1,0.15,0.8806,0.8310,0.8551,8,12,59
2,0.20,0.8939,0.8310,0.8613,7,12,59
3,0.25,0.9219,0.8310,0.8741,5,12,59
4,0.30,0.9219,0.8310,0.8741,5,12,59
5,0.35,0.9365,0.8310,0.8806,4,12,59
6,0.40,0.9365,0.8310,0.8806,4,12,59
7,0.45,0.9365,0.8310,0.8806,4,12,59
8,0.50,0.9365,0.8310,0.8806,4,12,59
9,0.55,0.9516,0.8310,0.8872,3,12,59


In [32]:
threshold_results_df.loc[
    threshold_results_df["F1"].idxmax()
]

Threshold           0.750000
Precision           0.967213
Recall              0.830986
F1                  0.893939
False Positives     2.000000
False Negatives    12.000000
True Positives     59.000000
Name: 13, dtype: float64

## Threshold Analysis

Threshold selection was evaluated on the validation set using the
class-weighted XGBoost model.

At a threshold of 0.75, the model achieved:
- Precision: 0.9672
- Recall: 0.8310
- F1-score: 0.8939
- False Positives: 2
- False Negatives: 12
- True Positives: 59

This threshold produced the highest F1-score among the tested
thresholds from 0.10 to 0.95. The threshold is treated as a
validation-stage candidate and will be reconsidered after model
hyperparameter tuning.

In [33]:
# Hyperparameter tuning using RandomizedSearchCV
tuning_configs = [
    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.1,
        "min_child_weight": 1,
    },
    {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.1,
        "min_child_weight": 1,
    },
    {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_child_weight": 1,
    },
    {
        "n_estimators": 500,
        "max_depth": 8,
        "learning_rate": 0.05,
        "min_child_weight": 1,
    },
    {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_child_weight": 5,
    }
]

print("Number of configurations:", len(tuning_configs))

Number of configurations: 5


In [34]:
#Run the experiments
tuning_results = []

for i, config in enumerate(tuning_configs, start=1):

    print(f"Training configuration {i}/{len(tuning_configs)}...")
    print(config)

    model = XGBClassifier(
        **config,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    val_proba = model.predict_proba(X_val)[:, 1]

    pr_auc_score = average_precision_score(
        y_val,
        val_proba
    )

    tuning_results.append({
        **config,
        "PR-AUC": pr_auc_score
    })

    print(f"PR-AUC: {pr_auc_score:.4f}")
    print("-" * 50)

Training configuration 1/5...
{'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1, 'min_child_weight': 1}
PR-AUC: 0.8699
--------------------------------------------------
Training configuration 2/5...
{'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'min_child_weight': 1}
PR-AUC: 0.8737
--------------------------------------------------
Training configuration 3/5...
{'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05, 'min_child_weight': 1}
PR-AUC: 0.8734
--------------------------------------------------
Training configuration 4/5...
{'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.05, 'min_child_weight': 1}
PR-AUC: 0.8740
--------------------------------------------------
Training configuration 5/5...
{'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05, 'min_child_weight': 5}
PR-AUC: 0.8742
--------------------------------------------------


In [35]:
tuning_results_df = pd.DataFrame(tuning_results)

tuning_results_df.sort_values(
    by="PR-AUC",
    ascending=False
).reset_index(drop=True)

,n_estimators,max_depth,learning_rate,min_child_weight,PR-AUC
0,500,6,0.05,5,0.874191
1,500,8,0.05,1,0.873956
2,300,6,0.10,1,0.873673
3,500,6,0.05,1,0.873398
4,300,4,0.10,1,0.869945


## Build the tuned model

In [36]:
best_params = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight,
    "random_state": 42,
    "eval_metric": "logloss",
    "n_jobs": -1
}

final_model = XGBClassifier(**best_params)

final_model.fit(
    X_train,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [37]:
y_val_proba_final = final_model.predict_proba(X_val)[:, 1]

print("Validation PR-AUC:",
      average_precision_score(y_val, y_val_proba_final))

Validation PR-AUC: 0.8741906972067733


In [38]:
thresholds = np.arange(0.10, 1.00, 0.05)

final_threshold_results = []

for threshold in thresholds:
    y_pred = (y_val_proba_final >= threshold).astype(int)

    final_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "False Positives": (
            (y_val == 0) & (y_pred == 1)
        ).sum(),
        "False Negatives": (
            (y_val == 1) & (y_pred == 0)
        ).sum(),
        "True Positives": (
            (y_val == 1) & (y_pred == 1)
        ).sum()
    })

final_threshold_df = pd.DataFrame(final_threshold_results)

final_threshold_df.round(4)

,Threshold,Precision,Recall,F1,False Positives,False Negatives,True Positives
0,0.10,0.7750,0.8732,0.8212,18,9,62
1,0.15,0.8108,0.8451,0.8276,14,11,60
2,0.20,0.8451,0.8451,0.8451,11,11,60
3,0.25,0.8429,0.8310,0.8369,11,12,59
4,0.30,0.8676,0.8310,0.8489,9,12,59
5,0.35,0.8939,0.8310,0.8613,7,12,59
6,0.40,0.8939,0.8310,0.8613,7,12,59
7,0.45,0.9077,0.8310,0.8676,6,12,59
8,0.50,0.9219,0.8310,0.8741,5,12,59
9,0.55,0.9365,0.8310,0.8806,4,12,59


In [39]:
best_threshold_row = final_threshold_df.loc[
    final_threshold_df["F1"].idxmax()
]

best_threshold_row

Threshold           0.850000
Precision           0.967213
Recall              0.830986
F1                  0.893939
False Positives     2.000000
False Negatives    12.000000
True Positives     59.000000
Name: 15, dtype: float64

In [40]:
final_threshold = 0.85

y_val_pred_final = (
    y_val_proba_final >= final_threshold
).astype(int)

final_val_pr_auc = average_precision_score(
    y_val,
    y_val_proba_final
)

final_val_precision = precision_score(
    y_val,
    y_val_pred_final
)

final_val_recall = recall_score(
    y_val,
    y_val_pred_final
)

final_val_f1 = f1_score(
    y_val,
    y_val_pred_final
)

final_val_cm = confusion_matrix(
    y_val,
    y_val_pred_final
)

print("===== FINAL VALIDATION RESULTS =====")
print(f"PR-AUC   : {final_val_pr_auc:.4f}")
print(f"Precision: {final_val_precision:.4f}")
print(f"Recall   : {final_val_recall:.4f}")
print(f"F1-score : {final_val_f1:.4f}")
print(f"Threshold: {final_threshold}")

print("\nConfusion Matrix:")
print(final_val_cm)

===== FINAL VALIDATION RESULTS =====
PR-AUC   : 0.8742
Precision: 0.9672
Recall   : 0.8310
F1-score : 0.8939
Threshold: 0.85

Confusion Matrix:
[[42486     2]
 [   12    59]]


## Final XGBoost Configuration

The final supervised model uses XGBoost with class weighting to address
the severe class imbalance.

Parameters:
- n_estimators: 500
- max_depth: 6
- learning_rate: 0.05
- min_child_weight: 5
- subsample: 0.8
- colsample_bytree: 0.8
- scale_pos_weight: 599.024
- random_state: 42

The validation PR-AUC was 0.8742.

A threshold of 0.85 was selected as the validation-stage threshold
based on the highest F1-score among the evaluated threshold grid.
At this threshold, validation precision was 0.9672, recall was
0.8310, and F1-score was 0.8939.

In [41]:
# Generate fraud probabilities on the untouched test set
y_test_proba = final_model.predict_proba(X_test)[:, 1]

print("Test probabilities generated.")
print("Min:", y_test_proba.min())
print("Max:", y_test_proba.max())

Test probabilities generated.
Min: 1.485132e-08
Max: 0.99999905


In [42]:
# Apply the threshold selected using validation data
y_test_pred = (
    y_test_proba >= final_threshold
).astype(int)

In [43]:
test_pr_auc = average_precision_score(
    y_test,
    y_test_proba
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_cm = confusion_matrix(
    y_test,
    y_test_pred
)

print("===== FINAL TEST RESULTS =====")
print(f"PR-AUC   : {test_pr_auc:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1-score : {test_f1:.4f}")
print(f"Threshold: {final_threshold}")

print("\nConfusion Matrix:")
print(test_cm)

===== FINAL TEST RESULTS =====
PR-AUC   : 0.8234
Precision: 0.9649
Recall   : 0.7746
F1-score : 0.8594
Threshold: 0.85

Confusion Matrix:
[[42486     2]
 [   16    55]]


In [44]:
import joblib
from pathlib import Path

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACTS_DIR / "xgboost_model.pkl"

joblib.dump(final_model, MODEL_PATH)

print(f"Model saved to: {MODEL_PATH}")

Model saved to: ..\artifacts\xgboost_model.pkl


In [45]:
import json

model_config = {
    "model_type": "XGBClassifier",
    "model_version": "xgboost_fraud_v1",

    "features": {
        "feature_count": len(X_train.columns),
        "feature_order": list(X_train.columns)
    },

    "hyperparameters": {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": float(scale_pos_weight),
        "random_state": 42,
        "eval_metric": "logloss"
    },

    "decision_threshold": final_threshold,

    "validation_metrics": {
        "pr_auc": float(final_val_pr_auc),
        "precision": float(final_val_precision),
        "recall": float(final_val_recall),
        "f1": float(final_val_f1)
    },

    "test_metrics": {
        "pr_auc": float(test_pr_auc),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1)
    }
}

CONFIG_PATH = ARTIFACTS_DIR / "model_config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(model_config, f, indent=4)

print(f"Config saved to: {CONFIG_PATH}")

Config saved to: ..\artifacts\model_config.json


In [46]:
metrics_report = pd.DataFrame([
    {
        "Model": "Baseline XGBoost",
        "PR-AUC": pr_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    },
    {
        "Model": "Weighted XGBoost",
        "PR-AUC": pr_auc_weighted,
        "Precision": precision_weighted,
        "Recall": recall_weighted,
        "F1": f1_weighted
    },
    {
        "Model": "Tuned XGBoost",
        "PR-AUC": final_val_pr_auc,
        "Precision": final_val_precision,
        "Recall": final_val_recall,
        "F1": final_val_f1
    },
    {
        "Model": "Final Test",
        "PR-AUC": test_pr_auc,
        "Precision": test_precision,
        "Recall": test_recall,
        "F1": test_f1
    }
])

metrics_report

,Model,PR-AUC,Precision,Recall,F1
0,Baseline XGBoost,0.867984,0.983051,0.816901,0.892308
1,Weighted XGBoost,0.873673,0.936508,0.830986,0.880597
2,Tuned XGBoost,0.874191,0.967213,0.830986,0.893939
3,Final Test,0.823433,0.964912,0.774648,0.859375


In [47]:
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = REPORTS_DIR / "xgboost_metrics.csv"

metrics_report.to_csv(METRICS_PATH, index=False)

print(f"Metrics report saved to: {METRICS_PATH}")

Metrics report saved to: ..\reports\xgboost_metrics.csv


In [48]:
confusion_matrices = {
    "validation": final_val_cm.tolist(),
    "test": test_cm.tolist()
}

CM_PATH = REPORTS_DIR / "xgboost_confusion_matrices.json"

with open(CM_PATH, "w") as f:
    json.dump(confusion_matrices, f, indent=4)

print(f"Confusion matrices saved to: {CM_PATH}")

Confusion matrices saved to: ..\reports\xgboost_confusion_matrices.json


## Final Model Summary

The final supervised fraud detection model is an XGBoost classifier
trained on the processed transaction features.

### Imbalance Handling

The training data is highly imbalanced, with approximately 599 legitimate
transactions for every fraudulent transaction. Class weighting was applied
using `scale_pos_weight`.

### Final Hyperparameters

- n_estimators: 500
- max_depth: 6
- learning_rate: 0.05
- min_child_weight: 5
- subsample: 0.8
- colsample_bytree: 0.8
- scale_pos_weight: approximately 599.024

### Threshold

A decision threshold of 0.85 was selected using the validation set based
on the highest F1-score among the evaluated threshold grid.

### Final Test Performance

- PR-AUC: 0.8234
- Precision: 0.9649
- Recall: 0.7746
- F1-score: 0.8594

The test set was kept separate from model selection and threshold tuning
and was used only for final evaluation.